# 08 — Census disparate-impact fairness audit (Chicago / NYC / LA)

- **Owner:** Bella  
- **Date:** 2026-07-12 (refreshed 2026-07-19)  
- **Models audited:** Model 1 (risk) — full battery; Model 2 (forecast) — calibration + coverage.  
- **Basis:** each city's chronological **test split** with realised labels (not the forward-looking `scores.json`).  
- **Artifacts written:** `reports/fairness/fairness_audit_<city>.json` + `reports/figures/fairness_*.png`.

> **Refresh note (2026-07-19):** Chicago reproduces exactly (it reads the frozen deployed feature snapshot); NYC and LA re-pull current SODA, so their row counts and numbers move as the data grows. On this run LA surfaces a new neighborhood false-positive-rate finding, so the cities are not on a single as-of date. See `docs/fairness_audit.md`.

> **The one rule:** census demographics are **audit-only** — they measure disparate impact and are never a model feature (decisions 0004 / 0005). This notebook joins them *after* the model, never before.

The audit reads a city-agnostic `AuditFrame` from a per-city adapter, joins ACS tract demographics (`foodsafety.audit.census`), and runs the metrics engine (`foodsafety.audit.fairness`): flag-rate parity, FPR, FNR, and calibration by group, each with bootstrap CIs and a material-and-confident verdict. See `src/foodsafety/audit/README.md` for the design.

Requires `CENSUS_API_KEY` in the environment and the `audit` extra (`uv sync --extra audit`).

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import average_precision_score

from foodsafety.audit import census, fairness, report, mitigation
from foodsafety.audit.adapters.chicago import ChicagoAdapter
from foodsafety.audit.adapters.nyc import NycAdapter
from foodsafety.audit.adapters.la import LaAdapter
from foodsafety.audit.census import ACS_YEAR

pd.set_option('display.width', 200, 'display.max_columns', 30)
OUT = Path('..') / 'reports' / 'fairness'
OUT.mkdir(parents=True, exist_ok=True)
ADAPTERS = {'chicago': ChicagoAdapter(), 'nyc': NycAdapter(), 'la': LaAdapter()}

## Build each city's audit frame, join census, and write the report
Each adapter reproduces its city's deployed model on the test split; the census join adds tract demographics; `report.build_report` assembles the reviewable JSON.

In [ ]:
reports = {}
for city, adapter in ADAPTERS.items():
    frame = adapter.build_audit_frame()
    frame = census.attach_area_demographics(frame, city=city)
    rep = report.build_report(frame, city, acs_year=ACS_YEAR)
    (OUT / f'fairness_audit_{city}.json').write_text(json.dumps(rep, indent=2))
    reports[city] = rep
    p = rep['provenance']
    print(f"{city:8s} rows={p['test_rows']:>6}  prevalence={p['label_prevalence']:.3f}  "
          f"M1 PR-AUC={p['model1_test_pr_auc']:.3f}  window={p['test_window']}")

## Visual summary of the fairness metrics
These figures read the freshly computed `reports` dict and are saved to `reports/figures/`. The scorecard is the quick read; the two close-ups explain the only two findings; the appendix keeps the full metric detail with confidence intervals. Colors are colorblind-safe and every finding carries a symbol as well as color.

In [ ]:
# Fairness-audit figures — colorblind-safe (Okabe-Ito); status is never
# color-alone (findings also carry a hatch and a marker). Figures are saved
# to reports/figures/ and displayed inline.
from pathlib import Path
FIGDIR = Path("..") / "reports" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

# --- style -----------------------------------------------------------------
CITY_ORDER = ["chicago", "nyc", "la"]
CITY_LABEL = {"chicago": "Chicago", "nyc": "New York City", "la": "Los Angeles"}

# Okabe-Ito colorblind-safe palette.
OK = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "sky": "#56B4E9",
    "purple": "#CC79A7",
    "grey": "#7F7F7F",
}
CLEAR = OK["blue"]        # within tolerance
FINDING = OK["vermillion"]  # material + CI-confident gap
INK = "#222222"
MUTED = "#6b6b6b"

# Human-readable axis names (the raw keys are terse).
AXIS_LABEL = {
    "neighborhood": "Neighborhood",
    "income": "Area income (quartile)",
    "race_nonwhite": "Area % non-white (quartile)",
    "race_dominant": "Area majority group",
    "poverty": "Area % in poverty (quartile)",
    "foreign_born": "Area % foreign-born (quartile)",
    "limited_english": "Area % limited-English (quartile)",
    "cuisine": "Cuisine",
    "tenure": "Establishment tenure",
    "facility_type": "Facility type",
}

LENS = [("fpr_gap", "False-positive-rate gap"),
        ("fnr_gap", "False-negative-rate gap"),
        ("ece_gap", "Calibration-error gap")]
LENS_COLOR = {"fpr_gap": OK["blue"], "fnr_gap": OK["green"], "ece_gap": OK["orange"]}


def _apply_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#cccccc",
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": "#e8e8e8",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "xtick.color": INK, "ytick.color": INK, "text.color": INK,
        "axes.labelcolor": INK,
        "svg.fonttype": "none",
    })


def _axis_label(key):
    return AXIS_LABEL.get(key, key.replace("_", " ").title())


def _gap(ax_data, metric, point="gaps_primary_high"):
    """Return the {value, ci_low, ci_high, finding, tolerance} entry for a metric."""
    for g in ax_data.get(point, []):
        if g["metric"] == metric:
            return g
    return None


# Short labels for the truth-conditioned metrics (used in annotations).
_METRIC_SHORT = {"fpr_gap": "false-positive", "fnr_gap": "false-negative", "ece_gap": "calibration"}


def fig_findings_by_lens(reports):
    """How many axes fire on the parity lens vs any truth-conditioned lens.

    Fully data-driven: the truth-conditioned bars are annotated with the exact
    axis + lens that fired, whatever the current data shows.
    """
    _apply_style()
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    truth = {"fpr_gap", "fnr_gap", "ece_gap"}
    parity_counts, truth_counts, totals, truth_notes = [], [], [], []
    for city in CITY_ORDER:
        axes = reports[city]["model1_risk"]["axes"]
        p = t = 0
        notes = []
        for key, a in axes.items():
            fired = {g["metric"] for g in a.get("gaps_primary_high", []) if g["finding"]}
            if "disparate_impact_ratio" in fired:
                p += 1
            hits = fired & truth
            if hits:
                t += 1
                for m in hits:
                    notes.append(f"{_axis_label(key)} ({_METRIC_SHORT[m]})")
        parity_counts.append(p)
        truth_counts.append(t)
        totals.append(len(axes))
        truth_notes.append(notes)
    y = np.arange(len(CITY_ORDER))
    h = 0.36
    ax.barh(y + h / 2, parity_counts, height=h, color=OK["sky"],
            edgecolor="white", label="Parity gap (flag rate differs)")
    ax.barh(y - h / 2, truth_counts, height=h, color=FINDING, hatch="///",
            edgecolor="white", label="Truth-conditioned gap (FPR / FNR / calibration)")
    for yi, (p, t, n, notes) in enumerate(zip(parity_counts, truth_counts, totals, truth_notes)):
        ax.text(p + 0.1, yi + h / 2, f"{p} of {n} axes", va="center", fontsize=9.5, color=MUTED)
        label = f"{t}" + (f"  ← {', '.join(notes)}" if notes else "")
        ax.text(t + 0.1, yi - h / 2, label, va="center", fontsize=9, color=FINDING if t else MUTED)
    ax.set_yticks(y)
    ax.set_yticklabels([CITY_LABEL[c] for c in CITY_ORDER])
    ax.set_xlabel("Number of demographic axes with a finding")
    ax.set_xlim(0, max(totals) + 4)
    ax.set_title("Where do fairness findings land? Parity fires broadly; the bias lenses rarely")
    ax.legend(loc="lower right", fontsize=9, frameon=False)
    fig.tight_layout()
    return fig


# --- Fig 1: parity ratio by axis -------------------------------------------
def fig_parity(reports):
    """Disparate-impact (four-fifths) ratio per axis, with bootstrap CI, per city."""
    _apply_style()
    fig, axs = plt.subplots(1, 3, figsize=(15, 5.2), sharex=True)
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        rows = []
        for key, a in axes.items():
            g = _gap(a, "disparate_impact_ratio")
            if g is None or g["value"] is None:
                continue
            rows.append((key, g["value"], g.get("ci_low"), g.get("ci_high"), g["finding"]))
        rows.sort(key=lambda r: r[1])  # worst parity at top
        labels = [_axis_label(r[0]) for r in rows]
        vals = [r[1] for r in rows]
        y = np.arange(len(rows))
        colors = [FINDING if r[4] else CLEAR for r in rows]
        hatches = ["///" if r[4] else "" for r in rows]
        bars = ax.barh(y, vals, color=colors, edgecolor="white", height=0.7)
        for b, hh in zip(bars, hatches):
            b.set_hatch(hh)
        # bootstrap CI whiskers
        for yi, r in enumerate(rows):
            lo, hi = r[2], r[3]
            if lo is not None and hi is not None:
                ax.plot([lo, hi], [yi, yi], color=INK, lw=1.1, alpha=0.6, zorder=5)
        # finding marker (non-color cue)
        for yi, r in enumerate(rows):
            if r[4]:
                ax.text(vals[yi] + 0.02, yi, "▲", va="center", ha="left",
                        fontsize=9, color=FINDING)
        ax.axvline(0.8, color=OK["vermillion"], ls="--", lw=1.3, zorder=1)
        ax.axvline(1.0, color=MUTED, ls=":", lw=1.1, zorder=1)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=9.5)
        ax.set_xlim(0, 1.12)
        ax.set_ylim(-0.7, len(rows) - 0.3)
        ax.set_title(CITY_LABEL[city])
        ax.set_xlabel("Disparate-impact ratio\n(lowest group flag rate ÷ highest)")
    fig.suptitle("Statistical parity by demographic axis  —  flagged = deployed High tier",
                 fontsize=13, fontweight="bold", y=0.99)
    fig.text(0.5, 0.93,
             "Shorter bar = larger flag-rate gap.   Dashed line = four-fifths rule (0.80);   "
             "dotted line = perfect parity (1.0).",
             ha="center", fontsize=9.5, color=MUTED)
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=CLEAR, label="Within four-fifths rule"),
               Patch(facecolor=FINDING, hatch="///", label="Parity finding (▲ below 0.80, CI-confident)")]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False,
               fontsize=10, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 0.88))
    return fig


# --- Fig 2: truth-conditioned lenses vs tolerance (forest plot) ------------
# A common axis-ordering so the three city panels line up row-for-row.
# Facility type is excluded: NYC and LA carry a single facility group, so the row is
# empty for two of the three cities and only adds noise to the comparison.
_LENS_AXIS_ORDER = ["neighborhood", "cuisine", "tenure",
                    "race_dominant", "race_nonwhite", "income", "poverty",
                    "foreign_born", "limited_english"]


def fig_bias_lenses(reports):
    """Forest plot: FPR / FNR / calibration gaps (as a fraction of their tolerance),
    each with its bootstrap CI. A finding only when the whole interval clears 1.0."""
    _apply_style()
    fig, axs = plt.subplots(1, 3, figsize=(15.5, 6.6))
    # One shared scale so the three cities are directly comparable. Anything past
    # XMAX is clipped to the edge and labelled with its true value, so a single big
    # outlier (NYC cuisine) can't squash every other dot against zero.
    XMAX = 2.2
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        keys = [k for k in _LENS_AXIS_ORDER if k in axes]
        y = np.arange(len(keys))

        # Row banding: alternating stripes make a row easy to follow across metrics.
        for yi in range(len(keys)):
            if yi % 2 == 0:
                ax.axhspan(yi - 0.5, yi + 0.5, color="#f4f6f8", zorder=0)
        # Over-tolerance zone.
        ax.axvspan(1.0, XMAX, color=OK["vermillion"], alpha=0.05, zorder=0)

        for j, (metric, _name) in enumerate(LENS):
            offs = (j - 1) * 0.26
            for yi, key in enumerate(keys):
                g = _gap(axes[key], metric)
                if g is None or g["value"] is None:
                    continue
                tol = g["tolerance"]
                val, lo, hi = g["value"] / tol, (g["ci_low"] or 0) / tol, (g["ci_high"] or 0) / tol
                ax.plot([min(lo, XMAX), min(hi, XMAX)], [yi + offs, yi + offs],
                        color=LENS_COLOR[metric], lw=2.0, alpha=0.6, zorder=3,
                        solid_capstyle="round")
                clipped = val > XMAX
                ax.scatter(min(val, XMAX), yi + offs,
                           s=70 if g["finding"] else 34,
                           marker=">" if clipped else "o",
                           color=LENS_COLOR[metric],
                           edgecolor=INK if g["finding"] else "white",
                           linewidth=1.3 if g["finding"] else 0.6,
                           zorder=5 if g["finding"] else 4)
                if clipped:
                    ax.text(XMAX - 0.04, yi + offs + 0.22, f"{val:.1f}×", va="center",
                            ha="right", fontsize=8, color=INK, fontweight="bold")
                if g["finding"]:
                    ax.text(min(hi, XMAX) - 0.04, yi + offs - 0.24, "▲ finding",
                            va="center", ha="right", fontsize=8, color=INK)
        ax.axvline(1.0, color=OK["vermillion"], ls="--", lw=1.4, zorder=2)
        ax.set_yticks(y)
        ax.set_yticklabels([_axis_label(k) for k in keys], fontsize=9.5)
        ax.set_ylim(-0.5, len(keys) - 0.5)
        ax.set_xlim(0, XMAX)
        ax.set_xticks([0, 0.5, 1.0, 1.5, 2.0])
        ax.set_title(CITY_LABEL[city])
        ax.set_xlabel("Gap ÷ its tolerance  (1.0 = the limit)")
        ax.grid(axis="y", visible=False)
    fig.suptitle("The bias lenses (false-positive, false-negative, calibration gaps) "
                 "vs their tolerance", fontsize=13, fontweight="bold", y=0.99)
    fig.text(0.5, 0.945, "A “gap” is the spread between the best and worst group on that "
             "check (see the legend below for what each one measures).",
             ha="center", fontsize=9.5, color=MUTED)
    fig.text(0.5, 0.905, "Dot = estimate, line = 95% CI, shaded band = past the limit.   "
             "A finding needs the whole interval past 1.0; a dot past it whose line crosses "
             "back is not confident.   ▶ = off scale, true value labelled.",
             ha="center", fontsize=9.5, color=MUTED)
    from matplotlib.lines import Line2D
    explain = {
        "fpr_gap": "False-positive-rate gap  ·  flagged High, but nothing went wrong",
        "fnr_gap": "False-negative-rate gap  ·  went wrong, but was not flagged",
        "ece_gap": "Calibration-error gap  ·  predicted risk vs what actually happened",
    }
    handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=LENS_COLOR[m],
                      markersize=9, label=explain[m]) for m, _n in LENS]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=10,
               bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 0.88))
    return fig


# --- Fig 3: prevalence tracking --------------------------------------------
def fig_prevalence_tracking(reports):
    """For parity-flagged axes: does the group flag rate track the real failure rate?"""
    _apply_style()
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    rows = []
    for city in CITY_ORDER:
        for key, a in reports[city]["model1_risk"]["axes"].items():
            g = _gap(a, "disparate_impact_ratio")
            if g and g["finding"] and a.get("flag_vs_prevalence_corr") is not None:
                rows.append((city, key, a["flag_vs_prevalence_corr"]))
    rows.sort(key=lambda r: r[2])
    y = np.arange(len(rows))
    corrs = [r[2] for r in rows]
    colors = [OK["green"] if c >= 0.5 else OK["orange"] for c in corrs]
    bars = ax.barh(y, corrs, color=colors, edgecolor="white", height=0.7)
    for yi, r in enumerate(rows):
        mk = "✓ tracks risk" if r[2] >= 0.5 else "○ weak/none"
        ax.text(max(r[2], 0) + 0.02, yi, mk, va="center", fontsize=8,
                color=OK["green"] if r[2] >= 0.5 else OK["orange"])
    ax.axvline(0.5, color=MUTED, ls="--", lw=1.2)
    ax.text(0.5, -0.9, "prevalence-tracking\nthreshold (0.5)", color=MUTED,
            fontsize=8.5, va="top", ha="center")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{CITY_LABEL[r[0]]}: {_axis_label(r[1])}" for r in rows], fontsize=9.5)
    ax.set_xlabel("Correlation of group flag rate with group failure rate")
    ax.set_xlim(min(0, min(corrs) - 0.1), 1.28)
    ax.set_ylim(-1.6, len(rows) - 0.3)
    ax.set_title("Do parity gaps track real risk?\n"
                 "High correlation = the model flags genuinely higher-risk areas, not bias",
                 fontsize=12)
    fig.tight_layout()
    return fig


# --- Fig 4: NYC cuisine calibration spotlight ------------------------------
def fig_nyc_cuisine(reports):
    """The one bias-lens finding: NYC calibration error across cuisines."""
    _apply_style()
    a = reports["nyc"]["model1_risk"]["axes"].get("cuisine")
    if a is None:
        return None
    gt = [r for r in a["group_table"] if r.get("audited") and r.get("ece") is not None]
    gt.sort(key=lambda r: r["ece"])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(14.5, 6.2))
    tol = reports["nyc"]["tolerances"]["ece_gap_max"]
    worst, best = gt[-1], gt[0]

    # left: ECE per cuisine vs tolerance; the max and min define the audited gap.
    y = np.arange(len(gt))
    bar_colors = [FINDING if r is worst else (OK["green"] if r is best else OK["sky"])
                  for r in gt]
    axl.barh(y, [r["ece"] for r in gt], color=bar_colors, edgecolor="white", height=0.72)
    axl.set_yticks(y)
    axl.set_yticklabels([r["group"] for r in gt], fontsize=9)
    axl.set_xlabel("Expected calibration error\n(|mean predicted − mean observed| risk)")
    axl.set_title("Calibration error by cuisine")
    axl.set_xlim(0, worst["ece"] * 1.35)
    axl.annotate("", xy=(worst["ece"], len(gt) - 1), xytext=(best["ece"], 0),
                 arrowprops=dict(arrowstyle="<->", color=FINDING, lw=1.3, ls=(0, (4, 3))))
    axl.text(worst["ece"] * 1.02, len(gt) / 2,
             f"across-group gap\n= {worst['ece'] - best['ece']:.3f}\n(tolerance {tol})",
             color=FINDING, fontsize=9, va="center")

    # right: reliability — predicted vs observed per cuisine, zoomed to the data.
    xs = [r["mean_pred"] for r in gt]
    ys = [r["mean_obs"] for r in gt]
    for r in gt:
        is_worst = r is worst
        axr.scatter(r["mean_pred"], r["mean_obs"],
                    s=max(35, r["n"] / 12), color=FINDING if is_worst else OK["blue"],
                    edgecolor="white", zorder=5, alpha=0.85)
    # Label only the notable points (extremes + biggest miscalibrations) to avoid clutter.
    notable = {worst["group"], best["group"]}
    for r in sorted(gt, key=lambda r: -abs(r["mean_obs"] - r["mean_pred"]))[:4]:
        notable.add(r["group"])
    # The beyond-chance cuisines sit in a tight cluster, so fan their labels out in
    # different directions instead of stacking them all up-and-right.
    fan = [(9, -14), (-11, 7), (9, 11), (-11, -16)]
    placed: list[tuple[float, float]] = []
    crowded_seen = 0
    for r in sorted((g for g in gt if g["group"] in notable), key=lambda g: -g["mean_obs"]):
        x, y = r["mean_pred"], r["mean_obs"]
        crowded = any(abs(px - x) < 0.05 and abs(py - y) < 0.03 for px, py in placed)
        if crowded:
            off = fan[crowded_seen % len(fan)]
            crowded_seen += 1
        else:
            off = (6, 4)
        axr.annotate(r["group"], (x, y), fontsize=8.5, color=INK, xytext=off,
                     textcoords="offset points", ha="left" if off[0] > 0 else "right")
        placed.append((x, y))
    lo = min(min(xs), min(ys)) - 0.03
    hi = max(max(xs), max(ys)) + 0.05
    axr.plot([lo, hi], [lo, hi], ls="--", color=MUTED, lw=1.2, zorder=1)
    axr.text(hi, hi, "perfect\ncalibration", color=MUTED, fontsize=8.5, ha="right", va="top")
    axr.annotate("above the line:\nmodel under-predicts risk", xy=(lo + 0.02, hi - 0.02),
                 fontsize=8.5, color=MUTED, va="top")
    axr.set_xlim(lo, hi)
    axr.set_ylim(lo, hi)
    axr.set_xlabel("Mean predicted risk")
    axr.set_ylabel("Mean observed failure rate")
    axr.set_title("Reliability by cuisine (marker size ∝ group size)")
    fig.suptitle("New York City cuisine: the audit's only calibration finding",
                 fontsize=13, fontweight="bold", y=0.99)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig



# ============================================================================
# Simplified, easier-to-read figures (added for readability pass).
# ============================================================================
from matplotlib.patches import Rectangle  # noqa: E402

# The three "bias" checks plus parity, in plain words for the scorecard columns.
_CHECKS = [
    ("disparate_impact_ratio", "Flags\nmore\noften", "parity"),
    ("fpr_gap", "False\nalarms\ndiffer", "bias"),
    ("fnr_gap", "Missed\nrisks\ndiffer", "bias"),
    ("ece_gap", "Score\naccuracy\ndiffers", "bias"),
]
_OKGREEN = "#3f9c6d"
_OKFILL = "#e5f2ea"
_DIFFILL = "#eceff2"
_FINDFILL = "#f6ddcb"


def fig_scorecard(reports):
    """One glanceable grid per city: which fairness checks pass, which flag.

    Parity differences are shown as a neutral dot (expected, tracks real risk);
    the three bias checks show a green tick (fine) or a red triangle (finding).
    """
    _apply_style()
    fig, axs = plt.subplots(1, 3, figsize=(15.5, 6.2))
    order = ["neighborhood", "facility_type", "cuisine", "tenure", "race_dominant",
             "race_nonwhite", "income", "poverty", "foreign_born", "limited_english"]
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        keys = [k for k in order if k in axes]
        nrow, ncol = len(keys), len(_CHECKS)
        for r, key in enumerate(keys):
            yy = nrow - 1 - r
            for c, (metric, _label, kind) in enumerate(_CHECKS):
                g = _gap(axes[key], metric)
                if g is None or g["value"] is None:
                    fill, sym, scol = "#f4f6f8", "–", MUTED   # en dash = not audited
                elif g["finding"] and kind == "bias":
                    fill, sym, scol = _FINDFILL, "▲", FINDING  # triangle
                elif g["finding"] and kind == "parity":
                    fill, sym, scol = _DIFFILL, "●", MUTED     # dot = flags more (expected)
                else:
                    fill, sym, scol = _OKFILL, "✓", _OKGREEN   # check = fine
                ax.add_patch(Rectangle((c, yy), 1, 1, facecolor=fill,
                                       edgecolor="white", linewidth=2))
                ax.text(c + 0.5, yy + 0.5, sym, ha="center", va="center",
                        fontsize=13, color=scol, fontweight="bold")
        ax.set_xlim(0, ncol)
        ax.set_ylim(0, nrow)
        ax.set_xticks([c + 0.5 for c in range(ncol)])
        ax.set_xticklabels([lbl for _, lbl, _ in _CHECKS], fontsize=8, linespacing=1.15)
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        ax.tick_params(length=0)
        ax.set_yticks([nrow - 1 - r + 0.5 for r in range(nrow)])
        ax.set_yticklabels([_axis_label(k) for k in keys], fontsize=9)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(CITY_LABEL[city], fontsize=12, pad=26)
        ax.grid(False)
    fig.suptitle("Fairness check results — almost everything passes; two findings",
                 fontsize=14, fontweight="bold", y=1.05)
    fig.text(0.5, 0.99, "Each row is a group type; each column is one fairness check.",
             ha="center", fontsize=10, color=MUTED)
    from matplotlib.patches import Patch
    handles = [
        Patch(facecolor=_OKFILL, edgecolor="white", label="✓  fine"),
        Patch(facecolor=_DIFFILL, edgecolor="white", label="●  flags this group more (expected – tracks real risk)"),
        Patch(facecolor=_FINDFILL, edgecolor="white", label="▲  finding (uneven errors – needs follow-up)"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False,
               fontsize=9.5, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 0.95), w_pad=4.0)
    return fig


def fig_nyc_cuisine_simple(reports):
    """Plain close-up of the NYC cuisine finding: predicted vs actual failure rate."""
    _apply_style()
    a = reports["nyc"]["model1_risk"]["axes"].get("cuisine")
    if a is None:
        return None
    from scipy.stats import binomtest

    gt = [r for r in a["group_table"] if r.get("audited") and r.get("ece") is not None]
    gt.sort(key=lambda r: r["ece"])
    worst = gt[-1]
    # Which cuisines deviate beyond chance? Under perfect calibration a group's
    # positives are Binomial(n, mean predicted); Bonferroni across the audited groups.
    thresh = 0.05 / len(gt)
    for r in gt:
        k = int(round(r["mean_obs"] * r["n"]))
        r["_p"] = binomtest(k, r["n"], r["mean_pred"], alternative="two-sided").pvalue
        r["_sig"] = r["_p"] < thresh
        r["_under"] = r["mean_obs"] > r["mean_pred"]
    n_sig = sum(r["_sig"] for r in gt)
    n_under = sum(r["_sig"] and r["_under"] for r in gt)

    fig, ax = plt.subplots(figsize=(9.5, 6.4))
    xs = [r["mean_pred"] for r in gt]
    ys = [r["mean_obs"] for r in gt]
    for r in gt:
        ax.scatter(r["mean_pred"], r["mean_obs"], s=max(40, r["n"] / 10),
                   color=FINDING if r["_sig"] else OK["blue"], edgecolor="white",
                   zorder=5, alpha=0.85)
    # Label every beyond-chance cuisine, plus the best-calibrated one for contrast.
    notable = {r["group"] for r in gt if r["_sig"]} | {gt[0]["group"], worst["group"]}
    # The beyond-chance cuisines sit in a tight cluster, so fan their labels out in
    # different directions rather than stacking them all up-and-right.
    fan = [(9, -15), (-11, 7), (9, 12), (-11, -17)]
    placed: list[tuple[float, float]] = []
    crowded_seen = 0
    for r in sorted((g for g in gt if g["group"] in notable), key=lambda g: -g["mean_obs"]):
        x, y = r["mean_pred"], r["mean_obs"]
        crowded = any(abs(px - x) < 0.05 and abs(py - y) < 0.03 for px, py in placed)
        if crowded:
            off = fan[crowded_seen % len(fan)]
            crowded_seen += 1
        else:
            off = (6, 4)
        ax.annotate(r["group"], (x, y), fontsize=9, color=INK, xytext=off,
                    textcoords="offset points", ha="left" if off[0] > 0 else "right")
        placed.append((x, y))
    lo = min(min(xs), min(ys)) - 0.03
    hi = max(max(xs), max(ys)) + 0.06
    ax.plot([lo, hi], [lo, hi], ls="--", color=MUTED, lw=1.3, zorder=1)
    ax.text(hi, hi, "  the model is right\n  (predicted = actual)", color=MUTED,
            fontsize=9, ha="right", va="top")
    ax.annotate("above the line:\nactual failures higher\nthan the model predicts",
                xy=(lo + 0.015, hi - 0.02), fontsize=9, color=FINDING, va="top", fontweight="bold")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Risk the model predicts")
    ax.set_ylabel("How often they actually fail")
    ax.set_title("New York City: are the risk scores accurate for each cuisine?\n"
                 f"{n_sig} deviate beyond chance (orange); {n_under} are under-predicted, "
                 "so their scores read safer than reality",
                 fontsize=12)
    fig.tight_layout()
    return fig


# LA county ZIP -> the USPS place name people actually use, so the axis reads as
# neighborhoods rather than five digits. Covers the ZIPs that surface in this chart;
# anything unmapped falls back to the bare ZIP.
_LA_ZIP_PLACE = {
    "90022": "East Los Angeles",
    "90201": "Bell Gardens / Cudahy",
    "90255": "Huntington Park",
    "90405": "Santa Monica",
    "90505": "Torrance",
    "90640": "Montebello",
    "91016": "Duarte",
    "91748": "Rowland Heights",
    "91754": "Monterey Park",
    "91776": "San Gabriel",
}


def _la_zip_label(z):
    place = _LA_ZIP_PLACE.get(str(z).strip())
    return f"{z}  {place}" if place else str(z)


def fig_la_neighborhood(reports):
    """Plain close-up of the LA finding: false-alarm rate differs by area (ZIP)."""
    _apply_style()
    a = reports["la"]["model1_risk"]["axes"].get("neighborhood")
    if a is None:
        return None
    gt = [r for r in a["group_table"] if r.get("audited") and r.get("fpr") is not None]
    gt.sort(key=lambda r: -r["fpr"])
    top = gt[:8]
    zero = sum(1 for r in gt if r["fpr"] == 0)

    # The "is this beyond sampling noise?" test (resampling every area at the pooled
    # rate) lives in scripts/run_fairness_followup_tests.py and is written up in
    # docs/fairness_audit.md; the chart just shows the measured rates.
    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    y = np.arange(len(top))[::-1]
    ax.barh(y, [r["fpr"] * 100 for r in top], color=FINDING, edgecolor="white", height=0.7)
    for yi, r in zip(y, top):
        ax.text(r["fpr"] * 100 + 0.2, yi, f"{r['fpr']*100:.0f}%  (n={r['n']})",
                va="center", fontsize=8.5, color=MUTED)
    ax.set_yticks(y)
    ax.set_yticklabels([_la_zip_label(r["group"]) for r in top], fontsize=9)
    ax.set_xlabel("False-alarm rate, %  (flagged High risk, but did not fail within 180 days)")
    ax.set_xlim(0, max(r["fpr"] for r in top) * 100 * 1.35)
    ax.set_title("Los Angeles: false-alarm rate by area (ZIP)\n"
                 f"The {len(top)} highest-rate areas, of {len(gt)} audited\n"
                 f"Across all {len(gt)}: rates run from "
                 f"{max(r['fpr'] for r in gt)*100:.0f}% down to 0%, and {zero} areas "
                 "had no false alarms at all", fontsize=12)
    fig.tight_layout()
    return fig


def fig_scorecard_detailed(reports):
    """The scorecard with the actual numbers, for anyone who wants them.

    Same grid as fig_scorecard (rows = group type, columns = the four checks) but each
    cell carries its measured value and is shaded by how close it sits to that check's
    own threshold, so "nearly over" is visible rather than hidden behind a green tick.
    Findings keep the outline + triangle. Coverage is appended on the right.
    """
    _apply_style()
    checks = [
        ("disparate_impact_ratio", "Flags\nmore\noften", "min"),
        ("fpr_gap", "False\nalarms", "max"),
        ("fnr_gap", "Missed\nrisks", "max"),
        ("ece_gap", "Score\naccuracy", "max"),
    ]
    order = ["neighborhood", "facility_type", "cuisine", "tenure", "race_dominant",
             "race_nonwhite", "income", "poverty", "foreign_born", "limited_english"]
    fig, axs = plt.subplots(1, 3, figsize=(17, 6.6))
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        keys = [k for k in order if k in axes]
        nrow, ncol = len(keys), len(checks) + 1  # +1 for the coverage column
        for r, key in enumerate(keys):
            yy = nrow - 1 - r
            for c, (metric, _lbl, direction) in enumerate(checks):
                g = _gap(axes[key], metric)
                degenerate = False
                if g is None or g["value"] is None:
                    fill, txt, tcol = "#f4f6f8", "–", MUTED
                else:
                    v, tol = g["value"], g["tolerance"]
                    # Severity, always "higher = worse". For the parity ratio the
                    # threshold is a FLOOR, so a ratio of 0 is the worst case, not the
                    # best: guard the divide instead of letting it fall through to 0.
                    if direction == "min":
                        sev = 2.0 if v <= 0 else tol / v
                    else:
                        sev = v / tol
                    shade = min(max(sev, 0.0), 2.0) / 2.0
                    fill = _severity_color(shade)
                    txt = f"{v:.2f}"
                    tcol = "#ffffff" if shade > 0.72 else INK
                    # A ratio of exactly 0 is DEGENERATE, not "infinitely unfair": it
                    # only means some audited group had zero High flags, which pins
                    # min/max to 0 regardless of how the other groups compare. Shading
                    # it as max severity overstates it, so mark it and mute it.
                    if direction == "min" and v <= 0:
                        # Plain grey, and NONE of the finding marks: the value carries
                        # no signal, so flagging it would say more than we measured.
                        fill, tcol, txt, degenerate = "#d4d8dd", INK, "0.00*", True
                ax.add_patch(Rectangle((c, yy), 1, 1, facecolor=fill,
                                       edgecolor="white", linewidth=2))
                ax.text(c + 0.5, yy + 0.5, txt, ha="center", va="center",
                        fontsize=9.5, color=tcol,
                        fontweight="bold" if g and g.get("finding") and not degenerate
                        else "normal")
                if g and g.get("finding") and not degenerate:
                    ax.add_patch(Rectangle((c + 0.04, yy + 0.04), 0.92, 0.92,
                                           fill=False, edgecolor=INK, linewidth=1.6))
                    ax.text(c + 0.93, yy + 0.86, "▲", ha="right", va="top",
                            fontsize=8, color=INK)
            cov = axes[key].get("coverage")
            ax.add_patch(Rectangle((ncol - 1, yy), 1, 1, facecolor="#ffffff",
                                   edgecolor="white", linewidth=2))
            ax.text(ncol - 0.5, yy + 0.5, "–" if cov is None else f"{cov * 100:.0f}%",
                    ha="center", va="center", fontsize=9, color=MUTED)
        ax.set_xlim(0, ncol)
        ax.set_ylim(0, nrow)
        ax.set_xticks([c + 0.5 for c in range(ncol)])
        ax.set_xticklabels([lbl for _, lbl, _ in checks] + ["Rows\naudited"],
                           fontsize=8.2, linespacing=1.2)
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        ax.tick_params(length=0)
        ax.set_yticks([nrow - 1 - r + 0.5 for r in range(nrow)])
        ax.set_yticklabels([_axis_label(k) for k in keys], fontsize=9)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(CITY_LABEL[city], fontsize=12, pad=30)
        ax.grid(False)
    fig.suptitle("Fairness scorecard, with the numbers", fontsize=14,
                 fontweight="bold", y=1.13)
    fig.text(0.5, 1.05,
             "Each cell is the measured value; darker = closer to (or past) that check's own "
             "threshold. Outlined + \u25b2 = a finding; a dash means too few rows to audit.\n"
             "Thresholds: flag-rate ratio must stay ABOVE 0.80; the false-alarm and "
             "missed-risk gaps below 0.10; the score-accuracy gap below 0.05.",
             ha="center", fontsize=9.5, color=MUTED)
    fig.text(0.5, 0.015,
             "* 0.00 (grey) is degenerate, not a measured extreme: one audited group had zero "
             "High flags, which forces the ratio to 0 no matter how the other groups "
             "compare. Read those rows from the false-alarm and missed-risk columns "
             "instead.",
             ha="center", fontsize=8.6, color=MUTED)
    fig.tight_layout(rect=(0, 0.05, 1, 0.93), w_pad=3.5)
    return fig


def _severity_color(t):
    """Light-to-dark single-hue ramp (0 = comfortably inside, 1 = at/past threshold)."""
    # sage -> deep terracotta, kept colorblind-safe by varying lightness strongly.
    stops = [(0.90, 0.94, 0.91), (0.99, 0.92, 0.80), (0.96, 0.74, 0.51), (0.84, 0.37, 0.09)]
    t = min(max(t, 0.0), 1.0) * (len(stops) - 1)
    i = int(t)
    if i >= len(stops) - 1:
        return stops[-1]
    f = t - i
    a, b = stops[i], stops[i + 1]
    return tuple(a[k] + (b[k] - a[k]) * f for k in range(3))


def show(fig, name):
    """Save a figure to reports/figures/ and return it for inline display."""
    if fig is not None:
        fig.savefig(FIGDIR / f"{name}.png", dpi=130, bbox_inches="tight")
    return fig


### The fairness scorecard
One row per group type, one column per fairness check. A green tick is fine; a grey dot means the model flags that group more (expected, and it tracks real risk - see the parity discussion below); an orange triangle is a finding to follow up.

In [ ]:
show(fig_scorecard(reports), 'fairness_01_scorecard')

#### The same scorecard, with the numbers
The pass/flag grid above is the quick read. This one keeps the same layout but shows each measured value, shaded by how close it sits to that check's own threshold, so a check that is *nearly* over is visible rather than hidden behind a green tick. The right-hand column is the share of rows the axis could audit.

In [ ]:
show(fig_scorecard_detailed(reports), 'fairness_05_scorecard_detailed')

### Finding 1 - New York City cuisine (score accuracy)
Each dot is a cuisine: the risk the model predicts (x) vs how often those places actually fail (y). On the dashed line the model is right. Bangladeshi sits well above it - those restaurants fail more often than the model predicts.

In [ ]:
show(fig_nyc_cuisine_simple(reports), 'fairness_02_nyc_cuisine')

### Finding 2 - Los Angeles neighborhood (false alarms)
The false-alarm rate (flagged High risk but did not fail within 180 days) by area (ZIP). A few small-count ZIPs sit well above the rest, which is why the gap clears the tolerance. Provisional: coarse LA geocoding means this may be noise.

In [ ]:
show(fig_la_neighborhood(reports), 'fairness_03_la_neighborhood')

### Appendix - detailed metrics with confidence intervals
For the record: every bias-lens gap (false-positive, false-negative, calibration) as a fraction of its tolerance, with 95% bootstrap intervals. A finding needs the whole interval past the line; a point estimate near the line whose interval crosses back is not a confident finding.

In [ ]:
show(fig_bias_lenses(reports), 'fairness_04_detail_bias_lenses')

## Per-city summary — which axes fire, on which lens
The key read: a **parity-only** finding (flag rate differs) is expected wherever true prevalence differs across groups — that is the model flagging higher-risk places, not bias. The lenses that catch bias are **FPR / FNR / calibration**, because they condition on the realised label.

In [ ]:
for city, rep in reports.items():
    print(f'==================== {city.upper()} ====================')
    print(pd.DataFrame(rep['model1_risk']['summary']).to_string(index=False))
    print()

## Verdict per axis (Model 1), with prevalence-tracking and the secondary operating point

In [ ]:
for city, rep in reports.items():
    print(f'-------------------- {city.upper()} --------------------')
    for key, ax in rep['model1_risk']['axes'].items():
        print(f"[{key}] corr={ax['flag_vs_prevalence_corr']}")
        print('   ', ax['verdict'])
    print()

## Model 2 (forecast) — calibration by group
The forecast has no flagging operating point, so only calibration + coverage are audited.

In [ ]:
for city, rep in reports.items():
    rows = [{'axis': k, 'coverage': a['coverage'], 'ece_gap': a['ece_gap'],
             'finding': a['finding']} for k, a in rep['model2_forecast']['axes'].items()]
    print(f'{city.upper()}:')
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## Mitigation cost (analysis only)
If an equalized-odds gap appeared, this prices the per-group thresholds that equalize recall. It does **not** change the model — adopting per-group thresholds is a scope call (Jun).

In [ ]:
chi = census.attach_area_demographics(ADAPTERS['chicago'].build_audit_frame(), city='chicago')
tbl = mitigation.equalize_recall_thresholds(chi, 'area_income_q')
print(tbl.to_string(index=False))
print('extra inspections to equalize recall across income quartiles:',
      tbl.attrs['total_delta_flags'], 'of', tbl.attrs['baseline_flagged'], 'flagged')